# Aprendizaje multimodal con MNIST + dígitos hablados

Un **modelo multimodal** combina información de varias fuentes (imagen, audio, texto…) que describen **lo mismo**. Aquí usaremos dos modalidades del mismo concepto, un dígito del 0 al 9:

| Modalidad | Dataset | Ejemplo |
|---|---|---|
| Imagen | **MNIST** (`tfds: mnist`) | dígito manuscrito 28×28 |
| Audio | **Free Spoken Digit Dataset** (`tfds: spoken_digit`) | una persona diciendo "seven" (8 kHz) |

Veremos dos formas muy distintas de usar varias modalidades:

1. **Late fusion (fusión tardía):** entrenamos un clasificador **por modalidad**, de forma independiente, y combinamos sus **predicciones** al final.
2. **CLIP (aprendizaje contrastivo):** entrenamos dos *encoders* para que proyecten la imagen y el audio del mismo dígito en un **espacio de embeddings compartido**, **sin usar las etiquetas como objetivo**. Después podemos clasificar y buscar entre modalidades (*zero-shot*, *retrieval*).

In [ ]:
# !pip install tensorflow tensorflow-datasets

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras import layers

print("TensorFlow:", tf.__version__)

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

#  Modalidad imagen: MNIST


In [ ]:
(ds_train_raw, ds_val_raw, ds_test_raw), ds_info = tfds.load(
    name="mnist",
    split=[
        "train[:40%]",
        "train[40%:50%]",
        "test[:50%]"
    ],
    shuffle_files=True,
    as_supervised=True,
    with_info=True
)

NUM_CLASSES = ds_info.features["label"].num_classes
CLASS_NAMES = [str(i) for i in range(NUM_CLASSES)]
print("Número de clases:", NUM_CLASSES)

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
BATCH_SIZE = 128
SHUFFLE_BUFFER = 10_000


def preprocess_mnist_train(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    image = tf.image.random_brightness(
        image,
        max_delta=0.10
    )
    image= tf.image.random_flip_left_right(image)
    image = tf.clip_by_value(image,0.0,1.0)
    label = tf.one_hot(label,depth=NUM_CLASSES)
    return image, label

def preprocess_eval_test(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    label = tf.one_hot(label,depth=NUM_CLASSES)
    return image, label



def build_dataset(dataset,preprocess_mnist=preprocess_eval_test, training=False):
    dataset = dataset.map(preprocess_mnist,num_parallel_calls=AUTOTUNE)
    if training:
        dataset = dataset.shuffle(
            SHUFFLE_BUFFER,
            seed=SEED,
            reshuffle_each_iteration=True)
    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(AUTOTUNE)
    return dataset

train_ds = build_dataset(ds_train_raw,preprocess_mnist=preprocess_mnist_train, training=True)
val_ds = build_dataset(ds_val_raw, preprocess_mnist=preprocess_eval_test, training=False)
test_ds = build_dataset(ds_test_raw, preprocess_mnist=preprocess_eval_test, training=False)

for images_batch, labels_batch in train_ds.take(1):
    print("Imágenes:", images_batch.shape, "| Etiquetas:", labels_batch.shape)

# Modalidad audio: Free Spoken Digit Dataset

`spoken_digit` contiene grabaciones WAV a **8 kHz** de 6 personas diciendo los dígitos 0–9 en inglés (50 repeticiones por dígito y persona). Cada archivo se llama `{dígito}_{persona}_{índice}.wav`.

**División oficial del dataset:** las repeticiones con índice 0–4 son **test**, 5–9 como **validación** y el resto (10–49) como **entrenamiento**. 
Las grabaciones tienen distinta duración; las recortamos o rellenamos con ceros a **1 segundo (8000 muestras)**, que cubre más del 99 % de los audios.

In [ ]:
ds_audio_raw, audio_info = tfds.load("spoken_digit", split="train", with_info=True)
print(audio_info.features)

SR = 8000         
AUDIO_LEN = SR   

waves, audio_labels, rec_index, speakers = [], [], [], []
for ex in tfds.as_numpy(ds_audio_raw):
    a = ex["audio"].astype(np.int16)[:AUDIO_LEN]
    waves.append(np.pad(a, (0, AUDIO_LEN - len(a))))
    audio_labels.append(int(ex["label"]))
    name = ex["audio/filename"]
    name = name.decode() if isinstance(name, bytes) else str(name)
    _, speaker, idx = name.split("/")[-1].replace(".wav", "").split("_")
    rec_index.append(int(idx)); speakers.append(speaker)

waves = np.stack(waves)        
audio_labels = np.array(audio_labels)
rec_index = np.array(rec_index)

is_test  = rec_index < 5
is_val   = (rec_index >= 5) & (rec_index < 10)
is_train = rec_index >= 10

waves_train, y_audio_train = waves[is_train], audio_labels[is_train]
waves_val,   y_audio_val   = waves[is_val],   audio_labels[is_val]
waves_test,  y_audio_test  = waves[is_test],  audio_labels[is_test]

print("Personas:", sorted(set(speakers)))
print(f"Audios -> train: {len(waves_train)} | val: {len(waves_val)} | test: {len(waves_test)}")
print("Audios por clase (train):", np.bincount(y_audio_train, minlength=NUM_CLASSES))

In [ ]:
from IPython import display
display.display(display.Audio(waves_train[1:2], rate=20000))
print(y_audio_train[1:2])

## Preprocesamiento de audio: lo mínimo posible

La forma de onda tiene 8000 muestras por segundo: como secuencia para una LSTM sería larguísima (8000 pasos) y cada paso tendría un solo número. Hacemos **una sola transformación**: partir la señal en ventanas cortas y calcular cuánta energía hay en cada frecuencia (el **espectrograma**).

```text
forma de onda int16 (8000,)
   ↓  ÷ 32768                           → valores en [-1, 1]
   ↓  STFT: ventanas de 256 muestras (32 ms) cada 128 muestras
   ↓  log(|·| + 0.001)                  → (61, 129)
espectrograma: 61 pasos de tiempo × 129 frecuencias
```

In [ ]:
FRAME_LEN, FRAME_STEP = 256, 128

def to_float(wave):
    return tf.cast(wave, tf.float32) / 32768.0                     # int16 -> [-1, 1]

def wave_to_spec(wave):
    stft = tf.signal.stft(wave, frame_length=FRAME_LEN, frame_step=FRAME_STEP)
    return tf.math.log(tf.abs(stft) + 1e-3)      

def time_shift(wave):
    shift = tf.random.uniform([], -800, 801, dtype=tf.int32)        # ±0.1 s
    return tf.roll(wave, shift, axis=-1)

def preprocess_audio_train(wave, label):
    return wave_to_spec(time_shift(to_float(wave))), tf.one_hot(label, depth=NUM_CLASSES)

def preprocess_audio_eval(wave, label):
    return wave_to_spec(to_float(wave)), tf.one_hot(label, depth=NUM_CLASSES)


specs_train = wave_to_spec(to_float(waves_train)).numpy()
SPEC_SHAPE = specs_train.shape[1:]
SPEC_MEAN = specs_train.mean(axis=(0, 1))
SPEC_VAR = specs_train.var(axis=(0, 1))
print("Forma del espectrograma (pasos, frecuencias):", SPEC_SHAPE)

In [ ]:
audio_train_ds = build_dataset(tf.data.Dataset.from_tensor_slices((waves_train, y_audio_train)),
                               preprocess_mnist=preprocess_audio_train, training=True)
audio_val_ds   = build_dataset(tf.data.Dataset.from_tensor_slices((waves_val, y_audio_val)),
                               preprocess_mnist=preprocess_audio_eval)
audio_test_ds  = build_dataset(tf.data.Dataset.from_tensor_slices((waves_test, y_audio_test)),
                               preprocess_mnist=preprocess_audio_eval)

for specs_batch, labels_batch in audio_train_ds.take(1):
    print("Espectrogramas:", specs_batch.shape, "| Etiquetas:", labels_batch.shape)

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(15, 5))
for d, ax_col in zip([0, 3, 5, 7, 9], axes.T):
    i = np.where(y_audio_train == d)[0][0]
    w = to_float(waves_train[i])
    ax_col[0].plot(np.arange(AUDIO_LEN) / SR, w.numpy(), lw=0.5)
    ax_col[0].set_title(f"Dígito {d}: forma de onda"); ax_col[0].set_xlabel("s")
    ax_col[1].imshow(wave_to_spec(w).numpy().T, origin="lower", aspect="auto", cmap="magma")
    ax_col[1].set_title("espectrograma"); ax_col[1].set_xlabel("paso de tiempo"); ax_col[1].set_ylabel("frecuencia")
plt.tight_layout(); plt.show()

#  Dataset multimodal: emparejar imagen y audio

Formamos pares **(imagen, audio)** eligiendo, para cada imagen, un audio **al azar de la misma clase** y de la **misma partición** (train con train, test con test), para no filtrar información entre particiones.


In [ ]:
def make_pairs(mnist_raw, waves_pool, labels_pool, seed):
    images, labels = [], []
    for img, lab in tfds.as_numpy(mnist_raw):
        images.append(img); labels.append(lab)
    images, labels = np.stack(images), np.array(labels)
    rng = np.random.default_rng(seed)
    pools = {c: np.where(labels_pool == c)[0] for c in range(NUM_CLASSES)}
    audio_idx = np.array([rng.choice(pools[c]) for c in labels])
    return images, waves_pool[audio_idx], labels

img_val_raw,  wav_val_pair,  y_val_pair  = make_pairs(ds_val_raw,  waves_val,  y_audio_val,  seed=1)
img_test_raw, wav_test_pair, y_test_pair = make_pairs(ds_test_raw, waves_test, y_audio_test, seed=2)


X_img_val,  _ = preprocess_eval_test(img_val_raw,  y_val_pair)
X_img_test, _ = preprocess_eval_test(img_test_raw, y_test_pair)
X_aud_val,  _ = preprocess_audio_eval(wav_val_pair,  y_val_pair)
X_aud_test, _ = preprocess_audio_eval(wav_test_pair, y_test_pair)

print("Pares de validación:", X_img_val.shape, X_aud_val.shape)
print("Pares de test:      ", X_img_test.shape, X_aud_test.shape)

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for k in range(8):
    axes[0, k].imshow(X_img_test[k].numpy().squeeze(), cmap="gray"); axes[0, k].set_title(f"y={y_test_pair[k]}")
    axes[1, k].imshow(X_aud_test[k].numpy().squeeze().T, origin="lower", aspect="auto", cmap="magma")
    for ax in axes[:, k]: ax.axis("off")
plt.suptitle("Pares (imagen, audio) del conjunto de test"); plt.tight_layout(); plt.show()

# Late fusion (fusión tardía)

```text
imagen ──► modelo imagen ──► p_img(y) ─┐
                                        ├──► regla de fusión ──► ŷ
audio  ──► modelo audio  ──► p_aud(y) ─┘
```
Cada modalidad usa la arquitectura que mejor encaja con su estructura:

- **Imagen → CNN 2D.** 
- **Audio → LSTM.** El espectrograma es una **secuencia**: 61 instantes, cada uno descrito por 129 frecuencias. 


In [ ]:
def build_image_body():
    return tf.keras.Sequential([
        tf.keras.Input(shape=(28, 28, 1)),
        layers.Conv2D(32, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(2),                                    
        layers.Conv2D(64, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(2),                            
        layers.Flatten(),
        layers.Dropout(0.30),
        layers.Dense(64, activation="relu"),
    ], name="image_body")

def build_audio_body():
    return tf.keras.Sequential([
        tf.keras.Input(shape=SPEC_SHAPE),            
        layers.Normalization(axis=-1, mean=SPEC_MEAN, variance=SPEC_VAR),
        layers.LSTM(64, return_sequences=True),
        layers.LSTM(64),
        layers.Dropout(0.30),
    ], name="audio_body")

def build_classifier(body, name):
    model = tf.keras.Sequential([body, layers.Dense(NUM_CLASSES, name="logits")], name=name)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
        metrics=[tf.keras.metrics.CategoricalAccuracy(name="accuracy")],
    )
    return model

def make_callbacks(patience=4):
    return [
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", mode="min", patience=patience, verbose=1)
    ]

tf.keras.backend.clear_session()
image_model = build_classifier(build_image_body(), "image_classifier")
audio_model = build_classifier(build_audio_body(), "audio_classifier")
image_model.summary()
audio_model.summary()

##  Entrenar cada modalidad por separado

In [ ]:
history_img = image_model.fit(train_ds, validation_data=val_ds, epochs=4,
                              callbacks=make_callbacks(patience=4), verbose=2)

In [ ]:
history_aud = audio_model.fit(audio_train_ds, validation_data=audio_val_ds, epochs=20,
                              callbacks=make_callbacks(patience=10), verbose=2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metric in zip(axes, ["loss", "accuracy"]):
    for name, h, c in [("imagen", history_img, "C0"), ("audio", history_aud, "C1")]:
        ax.plot(h.history[metric], color=c, label=f"{name} train")
        ax.plot(h.history[f"val_{metric}"], color=c, ls="--", label=f"{name} val")
    ax.set_title(metric); ax.set_xlabel("Época"); ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()

print("Test imagen (MNIST completo):", image_model.evaluate(test_ds, return_dict=True, verbose=0))
print("Test audio  (300 grabaciones):", audio_model.evaluate(audio_test_ds, return_dict=True, verbose=0))

## Reglas de fusión

Con las probabilidades de cada modelo, $p_{img}$ y $p_{aud}$:

| Regla | Fórmula | Comentario |
|---|---|---|
| Promedio | $\frac{1}{2}(p_{img} + p_{aud})$ | sin parámetros |
| Producto | $\log p_{img} + \log p_{aud}$ | equivale a suponer modalidades independientes dada la clase (Bayes) |
| Stacking | $\text{Dense}([z_{img}, z_{aud}])$ | un pequeño modelo aprende a combinar los logits; se entrena en **validación** |


In [ ]:
def predict_logits(model, X, batch_size=512):
    return model.predict(X, batch_size=batch_size, verbose=0)

logits_img_val,  logits_aud_val  = predict_logits(image_model, X_img_val),  predict_logits(audio_model, X_aud_val)
logits_img_test, logits_aud_test = predict_logits(image_model, X_img_test), predict_logits(audio_model, X_aud_test)

softmax = lambda z: tf.nn.softmax(z, axis=1).numpy()
acc = lambda p, y: float(np.mean(np.argmax(p, axis=1) == y))


In [ ]:
stack = tf.keras.Sequential([tf.keras.Input(shape=(2 * NUM_CLASSES,)),
                            layers.Dense(NUM_CLASSES)])
stack.compile(optimizer=tf.keras.optimizers.Adam(1e-2),
                loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True), 
                metrics=["accuracy"])
stack.fit(np.concatenate([logits_img_val, logits_aud_val], axis=1), y_val_pair,
            epochs=50, batch_size=256, verbose=1)

In [ ]:
def fuse_all(z_img, z_aud):
    p_img, p_aud = softmax(z_img), softmax(z_aud)
    return {
        "Solo imagen": p_img,
        "Solo audio": p_aud,
        "Promedio": 0.5 * (p_img + p_aud),
        "Producto": np.log(p_img + 1e-9) + np.log(p_aud + 1e-9),
        "Stacking": stack.predict(np.concatenate([z_img, z_aud], axis=1), verbose=0),
    }

In [ ]:
fused_test = fuse_all(logits_img_test, logits_aud_test)
results_fusion = {name: acc(p, y_test_pair) for name, p in fused_test.items()}
for name, a in results_fusion.items():
    print(f"{name:<22s} accuracy test = {a:.4f}")

In [ ]:
y_pred = np.argmax(fused_test["Producto"], axis=1)
cm = tf.math.confusion_matrix(y_test_pair, y_pred, num_classes=NUM_CLASSES).numpy()

plt.figure(figsize=(6, 5))
plt.imshow(cm, cmap="Blues"); plt.colorbar()
plt.xticks(range(NUM_CLASSES)); plt.yticks(range(NUM_CLASSES))
plt.xlabel("Predicción"); plt.ylabel("Clase real"); plt.title("Matriz de confusión — fusión por producto")
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        plt.text(j, i, cm[i, j], ha="center", va="center", fontsize=7)
plt.tight_layout(); plt.show()

#  CLIP: un espacio de embeddings compartido

**CLIP** (*Contrastive Language–Image Pre-training*, Radford et al., 2021) 

```text
imagen ─► image_body ─► Dense(d) ─► L2-normalizar ─► z_img ─┐
                                                              ├─► S = τ⁻¹ · z_img · z_audᵀ   (B × B)
audio  ─► audio_body ─► Dense(d) ─► L2-normalizar ─► z_aud ─┘
```

En un lote de $B$ pares, la matriz de similitudes $S$ tiene los pares correctos en la **diagonal**:

$$
\mathcal{L} = \tfrac{1}{2}\big[\text{CE}(S, I) + \text{CE}(S^\top, I)\big]
$$

$\tau$ es una **temperatura aprendible** (como en CLIP se aprende $\log(1/\tau)$, inicializada en $1/\tau = 1/0.07$ y acotada a 100).


## Pipeline de pares para entrenamiento

Para entrenar recorremos las imágenes de MNIST-train y, dentro del `map`, elegimos **en cada época un audio distinto** de la misma clase (los audios de train se guardan ordenados por clase). Así cada imagen ve audios distintos a lo largo del entrenamiento: es un aumento de datos gratuito. Cada modalidad usa su preprocesamiento de entrenamiento (`preprocess_mnist_train` y desplazamiento temporal del audio).

In [ ]:
order = np.argsort(y_audio_train, kind="stable")
WAVES_BY_CLASS = tf.constant(waves_train[order])           
CLASS_COUNT = tf.constant(np.bincount(y_audio_train, minlength=NUM_CLASSES), tf.int64)
CLASS_OFFSET = tf.constant(np.concatenate([[0], np.cumsum(np.bincount(y_audio_train, minlength=NUM_CLASSES))[:-1]]), tf.int64)

def make_clip_pair(image, label):
    image, _ = preprocess_mnist_train(image, label)
    label = tf.cast(label, tf.int64)
    j = CLASS_OFFSET[label] + tf.random.uniform([], 0, CLASS_COUNT[label], dtype=tf.int64)
    spec, _ = preprocess_audio_train(tf.gather(WAVES_BY_CLASS, j), label)
    return image, spec, label

CLIP_BATCH = 128
clip_train_ds = (ds_train_raw
                 .shuffle(SHUFFLE_BUFFER, seed=SEED)
                 .map(make_clip_pair, num_parallel_calls=AUTOTUNE)
                 .batch(CLIP_BATCH, drop_remainder=True)      # B fijo: la matriz S siempre es B×B
                 .prefetch(AUTOTUNE))

clip_val_ds = (tf.data.Dataset.from_tensor_slices((X_img_val, X_aud_val, y_val_pair))
               .batch(CLIP_BATCH, drop_remainder=True).prefetch(AUTOTUNE))

for xi, xa, yl in clip_train_ds.take(1):
    print("imágenes:", xi.shape, "| espectrogramas:", xa.shape, "| etiquetas:", yl.shape)

##  Encoders, temperatura y pérdida contrastiva

Reutilizamos los **mismos cuerpos** de la sección 4 (recién inicializados), pero en lugar de la cabeza `Dense(10)` ponemos una **cabeza de proyección** `Dense(d)` seguida de normalización L2.


In [ ]:
EMBED_DIM = 32

class L2Normalize(layers.Layer):
    def call(self, x):
        return tf.math.l2_normalize(x, axis=-1)

def build_encoder(body, name):
    return tf.keras.Sequential([body, layers.Dense(EMBED_DIM, name="projection"), L2Normalize()], name=name)

class CLIPModel(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.image_encoder = build_encoder(build_image_body(), "image_encoder")
        self.audio_encoder = build_encoder(build_audio_body(), "audio_encoder")
        self.log_inv_tau = self.add_weight(name="log_inv_tau", shape=(),
                                           initializer=tf.keras.initializers.Constant(np.log(1 / 0.07)))
        self.build_metrics()

    def scale(self):
        return tf.minimum(tf.exp(self.log_inv_tau), 100.0)

    def call(self, inputs, training=False):
        images, specs = inputs
        return self.image_encoder(images, training=training), self.audio_encoder(specs, training=training)

    def build_metrics(self):
        self.loss_tracker = tf.keras.metrics.Mean(name="loss")
        self.acc_tracker = tf.keras.metrics.Mean(name="acc_lote")

    @property
    def metrics(self):
        return [self.loss_tracker, self.acc_tracker]

    def _step(self, data, training):
        images, specs, labels = data
        z_img, z_aud = self((images, specs), training=training)
        loss, logits = clip_loss(z_img, z_aud, self.scale())
        nn = tf.argmax(logits, axis=1)
        self.acc_tracker.update_state(tf.cast(tf.gather(labels, nn) == labels, tf.float32))
        self.loss_tracker.update_state(loss)
        return loss

    def train_step(self, data):
        with tf.GradientTape() as tape:
            loss = self._step(data, training=True)
        grads = tape.gradient(loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))
        return {m.name: m.result() for m in self.metrics}

    def test_step(self, data):
        self._step(data, training=False)
        return {m.name: m.result() for m in self.metrics}



In [ ]:
def clip_loss(z_img, z_aud, scale):
    logits = scale * tf.matmul(z_img, z_aud, transpose_b=True)   # (B, B)
    targets = tf.range(tf.shape(logits)[0])                      # la imagen i va con el audio i
    loss_i2a = tf.keras.losses.sparse_categorical_crossentropy(targets, logits, from_logits=True)
    loss_a2i = tf.keras.losses.sparse_categorical_crossentropy(targets, tf.transpose(logits), from_logits=True)
    return tf.reduce_mean(loss_i2a + loss_a2i) / 2, logits

clip_model = CLIPModel()
clip_model.image_encoder.summary()

In [ ]:
clip_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3))

class TauLogger(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        logs["inv_tau"] = float(self.model.scale())
        
clip_history = clip_model.fit(clip_train_ds, validation_data=clip_val_ds, epochs=10,
                              callbacks=[TauLogger(),make_callbacks(patience=4)], verbose=2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(clip_history.history["loss"], label="train"); axes[0].plot(clip_history.history["val_loss"], label="val")
axes[0].set_title("Pérdida contrastiva")
axes[1].plot(clip_history.history["acc_lote"], label="train"); axes[1].plot(clip_history.history["val_acc_lote"], label="val")
axes[1].set_title("Recuperación por clase dentro del lote")
axes[2].plot(clip_history.history["inv_tau"]); axes[2].set_title("Temperatura aprendida (1/τ)")

Calculemos los embeddings de todos los pares de test. 

In [ ]:
Z_img_test = clip_model.image_encoder.predict(X_img_test, batch_size=512, verbose=0)
Z_aud_test = clip_model.audio_encoder.predict(X_aud_test, batch_size=512, verbose=0)

sel = np.concatenate([np.where(y_test_pair == c)[0][:6] for c in range(NUM_CLASSES)])
S = Z_img_test[sel] @ Z_aud_test[sel].T

plt.figure(figsize=(7, 6))
plt.imshow(S, cmap="viridis"); plt.colorbar(label="similitud coseno")
ticks = np.arange(3, len(sel), 6)
plt.xticks(ticks, range(NUM_CLASSES)); plt.yticks(ticks, range(NUM_CLASSES))
plt.xlabel("audio (ordenado por dígito)"); plt.ylabel("imagen (ordenada por dígito)")
plt.title("Similitud imagen–audio en test"); plt.show()

In [ ]:
from sklearn.decomposition import PCA
n_show = 1500
Z_all = np.concatenate([Z_img_test[:n_show], Z_aud_test[:n_show]])
labels_all = np.concatenate([y_test_pair[:n_show]] * 2)
pca = PCA(n_components=2)
P = pca.fit_transform(Z_all)

plt.figure(figsize=(8, 7))
for c in range(NUM_CLASSES):
    m_img = (labels_all == c) & (np.arange(len(P)) < n_show)
    m_aud = (labels_all == c) & (np.arange(len(P)) >= n_show)
    plt.scatter(*P[m_img].T, s=6, color=f"C{c}", alpha=0.4)
    plt.scatter(*P[m_aud].T, s=40, color=f"C{c}", marker="^", edgecolor="k", lw=0.3, label=str(c))
plt.legend(title="dígito (▲ = audio, • = imagen)", ncol=2, fontsize=8)
plt.title("Espacio compartido (PCA)"); plt.grid(alpha=0.3); plt.show()

##  Clasificación *zero-shot*

En CLIP, una imagen y  texto  como *"a photo of a dog"* se compara, ahora aquí  comparamos la imagen con un audio por dígito. Tambien al revés: clasificar audios con prototipos de imagen.

In [ ]:
def prototypes(Z, y):
    P = np.stack([Z[y == c].mean(axis=0) for c in range(NUM_CLASSES)])
    return P / np.linalg.norm(P, axis=1, keepdims=True)

# prototipos construidos con datos de VALIDACIÓN 
Z_aud_val_pool = clip_model.audio_encoder.predict(wave_to_spec(to_float(waves_val)), verbose=0)
Z_img_val_all  = clip_model.image_encoder.predict(X_img_val, batch_size=512, verbose=0)
proto_aud = prototypes(Z_aud_val_pool, y_audio_val)
proto_img = prototypes(Z_img_val_all, y_val_pair)

# imagen -> prototipos de audio (así suena un siete)
zs_img_pred = np.argmax(Z_img_test @ proto_aud.T, axis=1)
acc_zs_img = np.mean(zs_img_pred == y_test_pair)

# audio -> prototipos de imagen (“así se ve un siete”)
Z_aud_test_pool = clip_model.audio_encoder.predict(wave_to_spec(to_float(waves_test)), verbose=0)
zs_aud_pred = np.argmax(Z_aud_test_pool @ proto_img.T, axis=1)
acc_zs_aud = np.mean(zs_aud_pred == y_audio_test)

print(f"Zero-shot imagen → prototipos de audio:  accuracy = {acc_zs_img:.4f}  (supervisado: {results_fusion['Solo imagen']:.4f})")
print(f"Zero-shot audio  → prototipos de imagen: accuracy = {acc_zs_aud:.4f}")

In [ ]:
sims = Z_aud_test_pool @ Z_img_test.T            
top10 = np.argsort(-sims, axis=1)[:, :10]
precision_at_10 = np.mean(y_test_pair[top10] == y_audio_test[:, None])
print(f"Precisión@10 audio → imagen: {precision_at_10:.4f}")

queries = [np.where(y_audio_test == d)[0][0] for d in [1, 4, 7]]
fig, axes = plt.subplots(len(queries), 9, figsize=(15, 5.5))
for r, q in enumerate(queries):
    axes[r, 0].imshow(wave_to_spec(to_float(waves_test[q])).numpy().T,
                      origin="lower", aspect="auto", cmap="magma")
    axes[r, 0].set_title(f"consulta: audio '{y_audio_test[q]}'", fontsize=9)
    for k, idx in enumerate(top10[q][:8]):
        axes[r, k + 1].imshow(X_img_test[idx].numpy().squeeze(), cmap="gray")
        ok = y_test_pair[idx] == y_audio_test[q]
        axes[r, k + 1].set_title(f"{sims[q, idx]:.2f}", fontsize=9)
    for ax in axes[r]: ax.axis("off")
plt.suptitle("Audio → imágenes más similares")
plt.tight_layout() 
plt.show()